# G2P Simulated Error Dataset Builder

This notebook combines `data/g2p/manual_set_1.csv` and `data/g2p/manual_set_2.csv` into a single corpus, samples **50 sentences** as-is, and generates **50 paired sentences with simulated transcription errors** while strictly respecting the corpus **phoneme inventory**:

- **50 samples**: Original sentences kept as-is (`error_type = "none"`)
- **50 samples**: Based on those 50 sentences, with simulated errors:
  - **Stress misplacement** (`stress_misplacement`): 20 samples
  - **Vowel quality confusion** (`/e/` vs `/i/`, `/u/` vs `/o/`, `vowel_confusion`): 10 samples
  - **`/r/` `/l/` confusion** (`ɾ` vs `l` in the inventory, `r_l_confusion`): 5 samples
  - **Glottal stop deletion** (`ʔ` deletion, `glottal_stop_deletion`): 15 samples

Outputs (exported to `output/dictee/g2p_error_simulation/`):
- `sentences.txt`: 100 prompt lines in IPA phonemic notation for recording with [Dictée](https://github.com/benjaminbellamy/Dictee).
- `metadata.csv`: Full 100-sample metadata.


In [1]:
import random
from collections import Counter
from pathlib import Path

import pandas as pd

RANDOM_SEED = 42
rng = random.Random(RANDOM_SEED)

ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path(".").resolve()
DATA_DIR = ROOT / "data" / "g2p"
OUTPUT_DIR = ROOT / "output" / "dictee" / "g2p_error_simulation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root :", ROOT)
print("Data dir     :", DATA_DIR)
print("Output dir   :", OUTPUT_DIR.relative_to(ROOT))


Project root : /home/lorenzmarqueses/Documents/Git/transcription-correction
Data dir     : /home/lorenzmarqueses/Documents/Git/transcription-correction/data/g2p
Output dir   : output/dictee/g2p_error_simulation


## 1. Load Both CSV Files as a Single Corpus


In [2]:
df1 = pd.read_csv(DATA_DIR / "manual_set_1.csv")[["sentence", "phoneme"]].copy()
df1["source_file"] = "manual_set_1.csv"

df2 = pd.read_csv(DATA_DIR / "manual_set_2.csv")[["sentence", "phoneme"]].copy()
df2["source_file"] = "manual_set_2.csv"

corpus = pd.concat([df1, df2], ignore_index=True)

print(f"manual_set_1.csv: {len(df1):,} rows")
print(f"manual_set_2.csv: {len(df2):,} rows")
print(f"Combined corpus : {len(corpus):,} rows")
print(f"Null values     : {corpus.isna().sum().to_dict()}")
corpus.head(5)


manual_set_1.csv: 973 rows
manual_set_2.csv: 200 rows
Combined corpus : 1,173 rows
Null values     : {'sentence': 0, 'phoneme': 0, 'source_file': 0}


,sentence,phoneme,source_file
0,Ang salsitson ay masarap na palaman sa tinapay.,ʔaŋ salsi't͡ʃon ʔaj masa'ɾap na pala'man sa ti...,manual_set_1.csv
1,Mahalaga na ipagkaiba natin ang personal na op...,mahala'ɡa na ʔipaɡkaʔi'ba 'natin ʔaŋ peɾso'nal...,manual_set_1.csv
2,Ang nanay ay kumukuha ng mga gulay sa hardin p...,ʔaŋ 'nanaj ʔaj kumu'kuhaʔ naŋ ma'ŋa 'ɡulaj sa ...,manual_set_1.csv
3,Magandang alisan na tayo bago pa umulan ng mal...,maɡan'daŋ ʔa'lisan na 'tajo 'baɡo pa ʔumu'lan ...,manual_set_1.csv
4,Ang kanyang pamumuhay ay maluho dahil sa kanya...,ʔaŋ kan'jaŋ pamu'muhaj ʔaj ma'luho 'dahil sa k...,manual_set_1.csv


## 2. Inspect & Enforce the Phoneme Inventory

Key inventory conventions in this corpus:
- **Consonants**: `p b k ɡ t d ʔ m ŋ n ɾ f v s z ʃ ʒ h w j l` (note IPA `ɡ` U+0261 and alveolar tap `ɾ` U+027E, not ASCII `g` or `r`)
- **Vowels**: `a e i o u` (and `ə` in general inventory)
- **Affricates**: `t͡s d͡z t͡ʃ d͡ʒ` using tie bar `\u0361`
- **Primary stress**: `'` (U+0027), placed before the stressed syllable's onset
- **Glottal stop**: `ʔ` (U+0294)


In [3]:
CONS = set("pbkɡtdʔmŋnɾfvszʃʒhwjl")
VOW = set("aeiouə")
STRESS = "'"
TIE_BAR = "\u0361"
INVENTORY = CONS | VOW | {STRESS, TIE_BAR}

char_counts = Counter("".join(corpus["phoneme"].astype(str)))
out_of_inv = {ch: cnt for ch, cnt in char_counts.items() if ch not in INVENTORY and ch != " "}
print("Unique phoneme characters:", len(char_counts))
print("Out-of-inventory characters in raw corpus:", out_of_inv)


def is_inventory_valid(phoneme_str: str) -> bool:
    for word in phoneme_str.split():
        if not set(word).issubset(INVENTORY):
            return False
        if word.startswith(TIE_BAR) or word.endswith(TIE_BAR):
            return False
    return True


clean_corpus = corpus[corpus["phoneme"].apply(is_inventory_valid)].reset_index(drop=True)
print(f"Inventory-valid rows: {len(clean_corpus):,} / {len(corpus):,}")


Unique phoneme characters: 29
Out-of-inventory characters in raw corpus: {}
Inventory-valid rows: 1,173 / 1,173


## 3. Sample 50 Sentences As-Is


In [4]:
sampled = (
    clean_corpus.sample(n=50, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)
sampled["pair_id"] = range(len(sampled))

print("Sampled 50 sentences by source_file:")
print(sampled["source_file"].value_counts().to_dict())
sampled[["pair_id", "source_file", "sentence", "phoneme"]].head(5)


Sampled 50 sentences by source_file:
{'manual_set_1.csv': 42, 'manual_set_2.csv': 8}


,pair_id,source_file,sentence,phoneme
0,0,manual_set_1.csv,Maraming turista ang bumibisita sa alvarado ta...,ma'ɾamiŋ tu'ɾista ʔaŋ bumibi'sita sa ʔalva'ɾad...
1,1,manual_set_1.csv,Taoista ka ba?,ta'wista ka ba
2,2,manual_set_2.csv,Ang alpa ay nagbibigay ng payapang himig.,ʔaŋ 'ʔalpa ʔaj naɡbibi'ɡaj naŋ pa'japaŋ 'himiɡ
3,3,manual_set_2.csv,Ang layunin ng pagbubungkal ay upang lumabo an...,ʔaŋ la'junin naŋ paɡbubuŋ'kal ʔaj 'ʔupaŋ luma'...
4,4,manual_set_1.csv,Narinig ko ang gadang sa labas ng bahay.,naɾi'niɡ ko ʔaŋ 'ɡadaŋ sa la'bas naŋ 'bahaj


## 4. Define Inventory-Aware Error Simulators

Each simulator modifies a single word in the sentence's `phoneme` string while keeping all symbols strictly within `INVENTORY`:
1. **`simulate_stress_misplacement`**: Moves `'` to a different syllable onset in a multi-syllable stressed word.
2. **`simulate_vowel_confusion`**: Swaps `/e/` $\leftrightarrow$ `/i/` or `/u/` $\leftrightarrow$ `/o/`.
3. **`simulate_r_l_confusion`**: Swaps `ɾ` $\leftrightarrow$ `l` (using `ɾ` U+027E from the inventory, never ASCII `r`).
4. **`simulate_glottal_stop_deletion`**: Deletes a `ʔ` (U+0294) from a word.


In [5]:
VOWEL_MAP = {"e": "i", "i": "e", "u": "o", "o": "u"}
RL_MAP = {"ɾ": "l", "l": "ɾ"}


def syllable_starts(clean_word: str) -> list[int]:
    """Return character indices where each syllable begins in an unstressed word."""
    starts = []
    for i, ch in enumerate(clean_word):
        if ch not in VOW:
            continue
        if i == 0:
            starts.append(0)
        elif i >= 3 and clean_word[i - 2] == TIE_BAR:
            starts.append(i - 3)
        elif i == 2 and clean_word[0] in CONS and clean_word[1] in CONS:
            starts.append(0)
        elif clean_word[i - 1] in CONS:
            starts.append(i - 1)
        else:
            starts.append(i)
    return sorted(set(starts))


def can_stress_misplace(phoneme_str: str) -> bool:
    for w in phoneme_str.split():
        if w.count(STRESS) == 1 and sum(c in VOW for c in w) >= 2:
            clean_w = w.replace(STRESS, "")
            orig_idx = w.index(STRESS)
            if any(s != orig_idx for s in syllable_starts(clean_w)):
                return True
    return False


def simulate_stress_misplacement(phoneme_str: str, rng_obj: random.Random) -> tuple[str, str]:
    words = phoneme_str.split()
    candidates = []
    for wi, w in enumerate(words):
        if w.count(STRESS) == 1 and sum(c in VOW for c in w) >= 2:
            clean_w = w.replace(STRESS, "")
            orig_idx = w.index(STRESS)
            alt_starts = [s for s in syllable_starts(clean_w) if s != orig_idx]
            if alt_starts:
                candidates.append((wi, w, clean_w, alt_starts))
    wi, orig_w, clean_w, alt_starts = rng_obj.choice(candidates)
    new_idx = rng_obj.choice(alt_starts)
    new_w = clean_w[:new_idx] + STRESS + clean_w[new_idx:]
    words[wi] = new_w
    return " ".join(words), f"{orig_w} -> {new_w}"


def can_vowel_confuse(phoneme_str: str) -> bool:
    return any(c in VOWEL_MAP for c in phoneme_str)


def simulate_vowel_confusion(phoneme_str: str, rng_obj: random.Random) -> tuple[str, str]:
    words = phoneme_str.split()
    positions = [
        (wi, ci)
        for wi, w in enumerate(words)
        for ci, c in enumerate(w)
        if c in VOWEL_MAP
    ]
    wi, ci = rng_obj.choice(positions)
    orig_w = words[wi]
    new_w = orig_w[:ci] + VOWEL_MAP[orig_w[ci]] + orig_w[ci + 1:]
    words[wi] = new_w
    return " ".join(words), f"{orig_w} -> {new_w}"


def can_r_l_confuse(phoneme_str: str) -> bool:
    return any(c in RL_MAP for c in phoneme_str)


def simulate_r_l_confusion(phoneme_str: str, rng_obj: random.Random) -> tuple[str, str]:
    words = phoneme_str.split()
    positions = [
        (wi, ci)
        for wi, w in enumerate(words)
        for ci, c in enumerate(w)
        if c in RL_MAP
    ]
    wi, ci = rng_obj.choice(positions)
    orig_w = words[wi]
    new_w = orig_w[:ci] + RL_MAP[orig_w[ci]] + orig_w[ci + 1:]
    words[wi] = new_w
    return " ".join(words), f"{orig_w} -> {new_w}"


def can_delete_glottal(phoneme_str: str) -> bool:
    return "ʔ" in phoneme_str


def simulate_glottal_stop_deletion(phoneme_str: str, rng_obj: random.Random) -> tuple[str, str]:
    words = phoneme_str.split()
    positions = [
        (wi, ci)
        for wi, w in enumerate(words)
        for ci, c in enumerate(w)
        if c == "ʔ"
    ]
    wi, ci = rng_obj.choice(positions)
    orig_w = words[wi]
    new_w = orig_w[:ci] + orig_w[ci + 1:]
    words[wi] = new_w
    return " ".join(words), f"{orig_w} -> {new_w}"


## 5. Assign Error Types (20 / 10 / 5 / 15) & Build the 100-Sample Dataset

We assign the 50 sampled sentences to the 4 error types in order of rarity/constraint (`r_l_confusion`: 5, `vowel_confusion`: 10, `glottal_stop_deletion`: 15, `stress_misplacement`: 20), then combine the 50 as-is rows and 50 errored rows into the final 100-sample dataset.


In [6]:
ERROR_SPECS = [
    ("r_l_confusion", 5, can_r_l_confuse, simulate_r_l_confusion),
    ("vowel_confusion", 10, can_vowel_confuse, simulate_vowel_confusion),
    ("glottal_stop_deletion", 15, can_delete_glottal, simulate_glottal_stop_deletion),
    ("stress_misplacement", 20, can_stress_misplace, simulate_stress_misplacement),
]

available_indices = list(sampled.index)
rng.shuffle(available_indices)
assigned = {}

for err_type, target_n, checker, _ in ERROR_SPECS:
    eligible = [idx for idx in available_indices if checker(sampled.loc[idx, "phoneme"])]
    assert len(eligible) >= target_n, f"Not enough eligible rows for {err_type}: {len(eligible)} < {target_n}"
    chosen = eligible[:target_n]
    for idx in chosen:
        assigned[idx] = err_type
        available_indices.remove(idx)

assert len(assigned) == 50

simulators = {err_type: fn for err_type, _, _, fn in ERROR_SPECS}

clean_rows = pd.DataFrame({
    "pair_id": sampled["pair_id"],
    "source_file": sampled["source_file"],
    "sentence": sampled["sentence"],
    "phoneme": sampled["phoneme"],
    "reference_phoneme": sampled["phoneme"],
    "has_error": False,
    "error_type": "none",
    "error_detail": "",
})

error_records = []
for idx, row in sampled.iterrows():
    err_type = assigned[idx]
    err_phoneme, detail = simulators[err_type](row["phoneme"], rng)
    error_records.append({
        "pair_id": row["pair_id"],
        "source_file": row["source_file"],
        "sentence": row["sentence"],
        "phoneme": err_phoneme,
        "reference_phoneme": row["phoneme"],
        "has_error": True,
        "error_type": err_type,
        "error_detail": detail,
    })

error_rows = pd.DataFrame(error_records)
dataset = pd.concat([clean_rows, error_rows], ignore_index=True)
dataset.insert(0, "sample_id", range(len(dataset)))

print("Dataset shape:", dataset.shape)
print("\nBreakdown by has_error:")
print(dataset["has_error"].value_counts().to_dict())
print("\nBreakdown by error_type:")
print(dataset["error_type"].value_counts().to_dict())


Dataset shape: (100, 9)

Breakdown by has_error:
{False: 50, True: 50}

Breakdown by error_type:
{'none': 50, 'stress_misplacement': 20, 'glottal_stop_deletion': 15, 'vowel_confusion': 10, 'r_l_confusion': 5}


## 6. Validate & Export for Dictée

We validate inventory compliance and export:
1. `sentences.txt`: 100 IPA prompts (one per line) for recording in [Dictée](https://github.com/benjaminbellamy/Dictee).
2. `metadata.csv`: Full dataset metadata mapped 1-to-1 with prompt indices.


In [7]:
assert len(dataset) == 100, f"Expected 100 rows, got {len(dataset)}"
assert (dataset["has_error"] == False).sum() == 50
assert (dataset["has_error"] == True).sum() == 50
assert (dataset["phoneme"].loc[dataset["has_error"]] != dataset["reference_phoneme"].loc[dataset["has_error"]]).all()

invalid_rows = dataset[~dataset["phoneme"].apply(is_inventory_valid)]
assert len(invalid_rows) == 0, f"Found out-of-inventory phonemes:\n{invalid_rows}"

# 1. Save metadata.csv
METADATA_PATH = OUTPUT_DIR / "metadata.csv"
dataset.to_csv(METADATA_PATH, index=False, encoding="utf-8")

# 2. Save sentences.txt (IPA prompts for Dictée)
SENTENCES_PATH = OUTPUT_DIR / "sentences.txt"
with SENTENCES_PATH.open("w", encoding="utf-8") as f:
    for p in dataset["phoneme"]:
        f.write(f"{p}\n")

print("All 100 samples passed phoneme inventory validation.")
print(f"Saved metadata to : {METADATA_PATH.relative_to(ROOT)}")
print(f"Saved prompts to  : {SENTENCES_PATH.relative_to(ROOT)}")

dataset[dataset["has_error"]].groupby("error_type").head(2)[
    ["sample_id", "pair_id", "error_type", "error_detail", "phoneme"]
]


All 100 samples passed phoneme inventory validation.
Saved metadata to : output/dictee/g2p_error_simulation/metadata.csv
Saved prompts to  : output/dictee/g2p_error_simulation/sentences.txt


,sample_id,pair_id,error_type,error_detail,phoneme
50,50,0,glottal_stop_deletion,ʔaŋ -> aŋ,ma'ɾamiŋ tu'ɾista aŋ bumibi'sita sa ʔalva'ɾado...
51,51,1,stress_misplacement,ta'wista -> 'tawista,'tawista ka ba
52,52,2,stress_misplacement,pa'japaŋ -> paja'paŋ,ʔaŋ 'ʔalpa ʔaj naɡbibi'ɡaj naŋ paja'paŋ 'himiɡ
53,53,3,vowel_confusion,paɡbubuŋ'kal -> paɡbuboŋ'kal,ʔaŋ la'junin naŋ paɡbuboŋ'kal ʔaj 'ʔupaŋ luma'...
54,54,4,r_l_confusion,naɾi'niɡ -> nali'niɡ,nali'niɡ ko ʔaŋ 'ɡadaŋ sa la'bas naŋ 'bahaj
59,59,9,vowel_confusion,'koɾte -> 'koɾti,hin'diʔ naɡta'ɡal ʔaŋ paɡha'habol naŋ kani'laŋ...
60,60,10,glottal_stop_deletion,ʔapel'jido -> apel'jido,majɾo'ʔon ka'miŋ kapit'bahaj na ʔaŋ apel'jido ...
61,61,11,r_l_confusion,pinaɡbawa'lan -> pinaɡbawa'ɾan,pinaɡbawa'ɾan ʃa naŋ ɡu'ɾoʔ na batu'kin ʔaŋ ka...


## 7. Dictée Recording & Merging into Manifest JSONL

### Recording with Dictée
Launch Dictée pointing to the exported `sentences.txt`:
```bash
flatpak run fr.benjaminbellamy.dictee output/dictee/g2p_error_simulation/sentences.txt
```
Dictée records 16 kHz mono WAV files into `sentences/` (`audio_00.wav`..`audio_99.wav`) and maintains `trans.txt`.

### Merging into Manifest JSONL
Once audio is recorded, merge the WAV files and metadata into an ASR manifest (`manifest.jsonl`) using `scripts/merge_dictee.py`:
```bash
python scripts/merge_dictee.py \
    --dictee-dir output/dictee/g2p_error_simulation/sentences \
    --metadata output/dictee/g2p_error_simulation/metadata.csv \
    --output output/dictee/g2p_error_simulation/manifest.jsonl
```


In [8]:
import sys
sys.path.insert(0, str(ROOT))
from scripts.merge_dictee import merge_dictee_session

MANIFEST_PATH = OUTPUT_DIR / "manifest.jsonl"
recording_dir = OUTPUT_DIR / "sentences"

wav_files = list(recording_dir.glob("audio_*.wav")) if recording_dir.exists() else []
if wav_files:
    entries = merge_dictee_session(
        dictee_dir=recording_dir,
        metadata_path=OUTPUT_DIR / "metadata.csv",
        output_manifest_path=MANIFEST_PATH,
        project_root=ROOT,
    )
    print(f"Merged {len(entries)} recordings into: {MANIFEST_PATH.relative_to(ROOT)}")
else:
    print(f"Ready for Dictée recording at: {OUTPUT_DIR.relative_to(ROOT)}/sentences.txt")
    print(f"Once audio is recorded in {recording_dir.relative_to(ROOT)}, re-run this cell to generate manifest.jsonl.")


Ready for Dictée recording at: output/dictee/g2p_error_simulation/sentences.txt
Once audio is recorded in output/dictee/g2p_error_simulation/sentences, re-run this cell to generate manifest.jsonl.
